In [2]:
#!pip install faiss-cpu sentence-transformers
#!pip uninstall -y bitsandbytes
#!pip install -U bitsandbytes>=0.46.1


In [3]:
import faiss
import torch
from transformers import pipeline, BitsAndBytesConfig, AutoModelForCausalLM, AutoTokenizer
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [4]:
df_churn = pd.read_excel("CustomerChurn_processed.xlsx")

print(df_churn.info(), "\n")
print(df_churn.iloc[0], "\n")
print(df_churn.loc[0, "Summary"])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7032 entries, 0 to 7031
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 


# Vector Database + Embeddings

In [5]:
knowledge_passages = [
"Customer churn refers to the movement of customers from one service provider to another. In telecommunications markets, churn represents switching loyalty to a competing provider.",

"Churn prediction aims to identify customers who are likely to leave their service provider so that companies can take action to retain them.",

"Value-added services improve customer satisfaction and reduce the likelihood that customers will churn.",

"Improving service quality, customer support, and complaint handling can significantly reduce customer churn.",

"Customer dissatisfaction is one of the main causes of churn because customers often switch providers when their expectations are not met."
]

In [6]:
sentences = df_churn["Summary"].to_list() + knowledge_passages

model = SentenceTransformer('all-MiniLM-L6-v2') # retrieval model (sentence-BERT)

sentence_embeddings = model.encode(sentences).astype('float32')

faiss.normalize_L2(sentence_embeddings)

dimension = sentence_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(sentence_embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
query = """
Customer:
- Tenure: 1 month
- Contract: Month-to-month
- Payment method: Electronic check
- Internet service: DSL
- Online security: Disabled
- Tech support: Enabled

Predicted churn probability: 84.3%

Can you explain why this customer was predicted to churn?
"""

query_embedding = model.encode([query]).astype('float32')

faiss.normalize_L2(query_embedding)

# Retrieving Documents

In [8]:
k = 5 # number of instances to return
distances, indices = index.search(query_embedding, k)

context = ""

print("\n Top matching sentences:")

for i, idx in enumerate(indices[0]):

    print(f"{i+1}. {sentences[idx]} (Distance: {distances[0][i]:.4f})\n")
    context += sentences[idx] + "\n"


 Top matching sentences:
1. Churn prediction aims to identify customers who are likely to leave their service provider so that companies can take action to retain them. (Distance: 0.4881)

2. Customer 4723-BEGSG is a male who is not a senior citizen. He has a partner or dependents. He has been with the company for 72 month(s).
The customer has phone service and uses multiple lines.
He uses DSL internet service.
Online security is enabled, and online backup is not enabled. Device protection is enabled, and tech support is enabled.
Streaming TV is enabled, and streaming movies is enabled.
The customer is on a Two year contract and does not use paperless billing.
His payment method is Credit card (automatic). His monthly charge is 86.65 dollars and total charges so far are 6094.25 dollars.

The model predicted that this customer will churn with a probability of 3.17%.

Top Contributing factors (SHAP):
 tenure: -1.094
Contract: -1.036
PaymentMethod: -0.293
OnlineSecurity: -0.277
TechSuppo

# Model Prompt

In [9]:
print(context)

prompt = f"""
You are a customer churn analyst.

A machine learning model predicted the churn probability for a customer.

User query:
{query}

Retrieved example cases:
{context}
Important:
- The reference cases are examples of past customers.
- Use them to identify patterns related to churn.
- Do NOT assume the target customer has attributes that are not provided.
- Base your explanation only on the target customer details and the patterns seen in the reference cases.

Task:
Explain the churn prediction in simple terms.

Steps:
1. Identify important attributes of the target customer.
2. Compare them with patterns observed in the reference cases.
3. Explain how these factors influence churn risk.

Write a short, clear explanation grounded in the reference cases.
"""

Churn prediction aims to identify customers who are likely to leave their service provider so that companies can take action to retain them.
Customer 4723-BEGSG is a male who is not a senior citizen. He has a partner or dependents. He has been with the company for 72 month(s).
The customer has phone service and uses multiple lines.
He uses DSL internet service.
Online security is enabled, and online backup is not enabled. Device protection is enabled, and tech support is enabled.
Streaming TV is enabled, and streaming movies is enabled.
The customer is on a Two year contract and does not use paperless billing.
His payment method is Credit card (automatic). His monthly charge is 86.65 dollars and total charges so far are 6094.25 dollars.

The model predicted that this customer will churn with a probability of 3.17%.

Top Contributing factors (SHAP):
 tenure: -1.094
Contract: -1.036
PaymentMethod: -0.293
OnlineSecurity: -0.277
TechSupport: -0.230

Customer 6087-YPWHO is a male who is not

# Downloading LLM

In [10]:
quant_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4"
)

device = "cuda"

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [11]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config = quant_config,
    device_map = device
)

pipe = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

# Prompting

In [13]:
output = pipe(prompt,
              max_new_tokens = 300,
              #temperature = 0.5, # controls randomness for more varied outputs
              #top_p = 0.95, # controls how much prob mass is considered (top 95%)
              repetition_penalty = 1.2, # penalises repeating tokens
              do_sample = False,
              return_full_text = False, # prompt included
              #early_stopping = True, # stops generating when EOS token is reached
             pad_token_id = tokenizer.eos_token_id)

print(output[0]["generated_text"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:
Based on our analysis, the high churn probability for the given customer is mainly due to three key factors:

1. Short tenure: In most of the reference cases where customers had a similar low tenure as the target customer (only one month), they were more likely to churn. This suggests that new customers may be less satisfied or committed to the service.

2. Month-to-month contract: Customers with month-to-month contracts tend to have higher churn rates compared to those with longer-term contracts. It seems that having a long-term commitment reduces the likelihood of leaving the service.

3. Payment method: Using electronic checks as a payment method appears to increase the chance of churn, although it's worth noting that this factor contributed less than the other two mentioned above. Some studies suggest that customers who pay by credit card might feel more attached to the service provider because of the convenience and potential rewards offered through such payments.

In sum